In [2]:
import { ChatOpenAI } from "@langchain/openai";
import { ChatPromptTemplate } from "packages/deps.ts";

const openAiApiKey = Deno.env.get("OPENAI_API_KEY")

const model = new ChatOpenAI({
  model: "gpt-3.5-turbo",
  temperature: 0
});

In [11]:
import { z } from "zod";

const anecdote = z.object({anecdotes: z.array(z.object({
  titleText: z.string().describe("The title of the anecdote."),
  text: z.string().describe("The verbatim transcript of the anecdote"),
  descriptionText: z.number().describe("A summary of the anecdote."),
  filename: z.string().describe("The name of the file containing the anecdote."),
  mediaId: z.string().describe("The Media ID of the file containing the anecdote."),
  transcriptId: z.string().describe("The Transcript ID of the file containing the anecdote."),
  topics: z.string().describe("A comma-separated list of topics related to the anecdote."),
  rationale: z.string().describe("A rationale for the confidence rating."),
  confidence: z.number().describe("A floating point confidence rating from 0 to 1, where 0 doesn't relate to the prompt and 1 relates best."),
})),});

const structuredLlm = model.withStructuredOutput(anecdote, {
  method: "json_mode",
  name: "anecdote",
});

In [12]:
const docs = "You're nervous. I'm nervous. I don't like cameras. I've never been interviewed. She's a good editor. Oh, good. I need that. Edit me. Just edit me out. Okay, let's go. Can you just back attack surfaces? So at Hany and company, the tax services we office our. Wow. Okay. Restart that. So at Haney and company, the tax services that we offer. My goodness. At Haney and company, the tax services that we offer our encompassing of the client's entire business, we really like to look at their entire business and work through their whole tax planning strategy. We focus on the tax preparation, whether it be a business, an individual, an estate trust, or nonprofit. We provide tax preparation services along all those service lines. We also provide state and federal tax planning services, and state and federal and local tax consulting services as well. I looked at them like an hour ago. You can do like the. You can do that like the planning one. And. Yeah, the tax planning. What tax planning services does. So at Haney and company, the tax planning services that we specialize in, I really like to look at it in a two step approach. We have the current year tax planning services where we like to look at, you know, minimizing our taxable income, whether that be, you know, maximizing the credits that we can take in this year, maximizing the deductions, and just making sure that we're overall built well for this tax year, and then also looking into the future. So making sure that we are set up in the right lines from a tax structure for our entities and, you know, looking at our retirement opportunities to make sure that we're making sure that we're taking advantage of the correct retirement planning opportunities from a tax standpoint and just really encompassing and looking at the entire business, just to make sure that we're building everything correctly and asking the client the questions, to make sure that we're building a plan for their needs, to really build them for the future and what their end goal is. What can clients expect from hanging company in terms of support? Audited by the IR's. So if audited by the IR's, what a client can expect from hanging company is really for us to be there with them for every step of the way through the process. Obviously, that can be a scary time to be audited by the IR's. And we really just want to be there to support them from the beginning to the end, to consult them on what they need to do and what strategies they need to take, you know, to best get through the situation and best handle the situation. I like how you started. What do clients expect from haitian company in terms of support? If audited by the IR's. So if audited by the IR's, what a client can expect from haitian company is really for us to be there throughout the entire process and through every step of the way to make sure we're providing advice to them and really make sure we're supporting them. It can obviously be a scary time for a client to go through a situation like that. And we want to provide the services that we can really to answer all their questions and make sure all their needs are met and really just make sure they make it through. And it really is not as bad as it seems. A lot of times IR's audits don't end up resulting in anything, and clients are just scared of the situation. And once we handle it, everything's taken care of fine, I can do that. What? Citadium, part one so what sets Haney and company apart from other CPA firms in henning complex tax situations really is, I think a lot of it is the size of the firm that we are. So we're a larger regional firm. We have the resources and the capabilities of any large firm. And really just the overall people that we have. We have so many resources, obviously, with how complex the tax code is. You know, not everybody can specialize in everything, but with the professionals that we have here, there has to be somebody that's ran into it at some point in time. So for us to be able to pull those resources together and really help those clients through those situations, and then also while being a larger firm, we still have that smaller firm approach where we develop relationships with our clients, and we're able to holistically look at their business and consult with them and really know where they want to be in the future and just really be a part of their day to day business. And we pride ourselves on being trusted advisors to our clients, and this enables us to do that. So Haney and company provides a fantastic work life balance. Obviously, the CPA industry is tough when it comes to that, and a lot of firms struggle with that. But Haney and company really sets itself apartheid by prioritizing families. You know, we're a family firm first, and we truly mean that. During the off season, we promote Haney Fridays, in which our staff, as long as all their work's been completed, their staff can take Fridays off and, you know, really spend that time with their family. And, yeah, sorry, I had things I want to say. And then, like, when I'm like, yeah. What do you value about working at Haney company? So what I value about working at hainian companies is the family firm approach. I'm a remote employee, and obviously being a remote employee, it's harder to build relationships with the staff. But everybody here is so open and friendly, being almost 500 employees, and I've met a lot of people through my two years that I've been here. And just the overall friendliness of the people that we work with and the value that they put towards the business and the passion that they have towards this industry really sets us apart. And thanks."

In [13]:
const createSystemMessage = (docs) => {

  return `
You have access to several video transcripts. Your task is to extract all anecdotes from these transcripts based on a user-provided prompt. This task is to be performed using the provided transcript data sections listed below. 

Each anecdote should:
- Be directly relevant to the specified word or concept wherever it appears in the transcripts.
- Have a clear beginning and end, focusing on distinct narratives within the text.
- Be a coherent standalone story or joke.
- Be verbatim text from the transcript

It is crucial that the language of the output matches the language of the input.

Ensure that the anecdotes are directly related to the user-provided word or concept. Avoid metaphors, analogies, or abstract interpretations. Focus strictly on direct mentions and explicit contexts related to the user-provided word or concept. 


Here are the video transcripts to reference:
${docs}
    `;
};

In [15]:
const prompt = ChatPromptTemplate.fromMessages([
    ["system", `${createSystemMessage(docs)}`],
    ["user", "{input}"],
  ]);

  const chain = prompt.pipe(structuredLlm);
  const start = performance.now();
  const response = await chain.invoke({ input: "comical moments" });
  const end = performance.now();
  const duration = end - start;
console.log(response, (duration/1000))

{
  anecdotes: [
    {
      titleText: "Nervous Interview",
      text: "You're nervous. I'm nervous. I don't like cameras. I've never been interviewed. She's a good editor. Oh, good. I need that. Edit me. Just edit me out. Okay, let's go.",
      descriptionText: 1,
      filename: "transcript1.txt",
      mediaId: "1",
      transcriptId: "1",
      topics: "comical moments",
      rationale: "The anecdote captures a humorous interaction about being nervous before an interview and the desire to be edited out.",
      confidence: 0.9
    },
    {
      titleText: "Tax Planning Services",
      text: "I looked at them like an hour ago. You can do like the. You can do that like the planning one. And. Yeah, the tax planning.",
      descriptionText: 2,
      filename: "transcript1.txt",
      mediaId: "1",
      transcriptId: "1",
      topics: "comical moments",
      rationale: "The anecdote presents a comical moment of confusion or distraction during a discussion about tax planning s